# Segmentación semántica: CamVid

Carga de 701 escenas urbanas con máscaras de 11 clases. En orden de índice, de 0 a 10: cielo, edificio, poste, carretera, banqueta, árbol, señal, cerca, automóvil, peatón y ciclista. El valor 11 representa píxeles sin etiqueta. Se conservan las particiones originales: 367 de entrenamiento, 101 de validación y 233 de prueba.

Esta libreta se limita a descargar, localizar y cargar imágenes y máscaras. Los archivos se almacenan en el entorno de ejecución.

[Fuente de los datos](https://github.com/alexgkendall/SegNet-Tutorial/tree/master/CamVid).


## Dependencias

Ejecute las celdas en orden en Colab. Se requieren Pillow, NumPy y Matplotlib.


In [ ]:
%pip -q install pillow numpy matplotlib


## Descarga y lectura

Las descargas incompletas no se reutilizan. Los datos se extraen una sola vez y después se comprueba que cada imagen tenga su máscara.


In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import shutil
import stat
import tarfile
import time
import urllib.request
import zipfile
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

RAIZ = Path('/content/datos_segmentacion') if Path('/content').exists() else Path('datos_segmentacion')
RAIZ.mkdir(parents=True, exist_ok=True)

def huella_md5(ruta):
    h = hashlib.md5()
    with ruta.open('rb') as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()

def descargar(urls, destino, md5=None):
    """Guarda solo descargas completas; reutiliza el archivo verificado."""
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    sello = destino.with_name(destino.name + '.completo.json')
    if destino.exists() and sello.exists():
        esperado = json.loads(sello.read_text())
        if destino.stat().st_size == esperado['bytes'] and (md5 is None or huella_md5(destino) == md5):
            print('Descarga disponible:', destino.name)
            return destino
    errores = []
    for url in urls:
        temporal = destino.with_name(destino.name + '.part')
        try:
            print('Descargando:', url, flush=True)
            solicitud = urllib.request.Request(url, headers={'User-Agent': 'segmentation-notebook/1.0'})
            with urllib.request.urlopen(solicitud, timeout=45) as respuesta, temporal.open('wb') as salida:
                longitud = respuesta.headers.get('Content-Length')
                recibidos = 0
                reporte = time.monotonic()
                while True:
                    bloque = respuesta.read(1024 * 1024)
                    if not bloque:
                        break
                    salida.write(bloque)
                    recibidos += len(bloque)
                    if time.monotonic() - reporte > 5:
                        print(f'  {recibidos / 1024**2:.1f} MiB recibidos', flush=True)
                        reporte = time.monotonic()
            if not recibidos or (longitud and recibidos != int(longitud)):
                raise RuntimeError(f'Descarga incompleta: {recibidos} bytes; se esperaban {longitud}.')
            if md5 and huella_md5(temporal) != md5:
                raise RuntimeError('El checksum no coincide con la fuente.')
            temporal.replace(destino)
            sello.write_text(json.dumps({'url': url, 'bytes': recibidos, 'md5': md5}))
            print('Descarga completa:', destino.name)
            return destino
        except Exception as error:
            errores.append(f'{url}: {error}')
            if temporal.exists():
                temporal.unlink()
    raise RuntimeError('No fue posible descargar el archivo. Revise la conexión o las fuentes:\n' + '\n'.join(errores))

def extraer(archivo, destino, prefijo=None):
    """Extrae archivos normales, sin rutas externas ni enlaces simbólicos."""
    destino = Path(destino)
    destino.mkdir(parents=True, exist_ok=True)
    sello = destino / '.extraccion_completa.json'
    identidad = {'archivo': archivo.name, 'bytes': archivo.stat().st_size, 'prefijo': prefijo}
    if sello.exists() and json.loads(sello.read_text()) == identidad:
        return destino
    def ruta_segura(nombre):
        ruta = (destino / nombre).resolve()
        if not ruta.is_relative_to(destino.resolve()):
            raise RuntimeError(f'Ruta no válida en el archivo: {nombre}')
        ruta.parent.mkdir(parents=True, exist_ok=True)
        return ruta
    if zipfile.is_zipfile(archivo):
        with zipfile.ZipFile(archivo) as z:
            for item in z.infolist():
                if item.is_dir() or (prefijo and prefijo not in item.filename):
                    continue
                if stat.S_ISLNK(item.external_attr >> 16):
                    raise RuntimeError('El ZIP contiene un enlace simbólico inesperado.')
                with z.open(item) as entrada, ruta_segura(item.filename).open('wb') as salida:
                    shutil.copyfileobj(entrada, salida)
    elif tarfile.is_tarfile(archivo):
        with tarfile.open(archivo) as tar:
            for item in tar:
                if not item.isfile():
                    continue
                with tar.extractfile(item) as entrada, ruta_segura(item.name).open('wb') as salida:
                    shutil.copyfileobj(entrada, salida)
    else:
        raise RuntimeError(f'No se encontró un ZIP o TAR válido en {archivo}.')
    sello.write_text(json.dumps(identidad))
    print('Extracción completa:', destino.name)
    return destino

def unica(raiz, nombre):
    candidatos = [p for p in raiz.rglob(nombre) if '__MACOSX' not in p.parts and not p.name.startswith('._')]
    if len(candidatos) != 1:
        raise RuntimeError(f'Se esperaba una ruta {nombre!r} en {raiz}; se encontraron {len(candidatos)}.')
    return candidatos[0]

def verificar_pares(registros, esperado=None):
    if not registros or (esperado is not None and len(registros) != esperado):
        raise RuntimeError(f'Se esperaban {esperado} pares; se encontraron {len(registros)}.')
    ids = [r['id'] for r in registros]
    if len(ids) != len(set(ids)):
        raise RuntimeError('Hay identificadores repetidos; revise la estructura de datos.')
    faltantes = [str(r[k]) for r in registros for k in ('imagen','mascara') if not Path(r[k]).is_file()]
    if faltantes:
        raise RuntimeError(f'Faltan {len(faltantes)} archivos. Primeros: {faltantes[:3]}')
    print('Pares comprobados:', len(registros))

def leer_par(registro):
    with Image.open(registro['imagen']) as im:
        imagen = im.convert('RGB').copy()
    with Image.open(registro['mascara']) as m:
        mascara = m.copy()  # Conservar índices de clase; no convertir a RGB.
    if imagen.size != mascara.size:
        raise RuntimeError(f"Dimensiones incompatibles en {registro['id']}: {imagen.size} y {mascara.size}.")
    return imagen, mascara


In [ ]:
CARPETA = RAIZ / 'camvid'
archivo = descargar(
    ['https://github.com/alexgkendall/SegNet-Tutorial/archive/refs/heads/master.zip',
     'https://codeload.github.com/alexgkendall/SegNet-Tutorial/zip/refs/heads/master'],
    CARPETA / 'segnet_tutorial.zip',
)
contenido = extraer(archivo, CARPETA / 'extraido', prefijo='/CamVid/')
raiz_camvid = unica(contenido, 'CamVid')
registros = []
for particion, cantidad in [('train', 367), ('val', 101), ('test', 233)]:
    lineas = (raiz_camvid / f'{particion}.txt').read_text().splitlines()
    pares = [l.split() for l in lineas if l.strip()]
    if len(pares) != cantidad or any(len(par) != 2 for par in pares):
        raise RuntimeError(f'Lista {particion} inesperada: {len(pares)} pares; se esperaban {cantidad}.')
    for imagen, mascara in pares:
        # Las listas originales usan /SegNet/CamVid/...; aquí se reconstruye la ruta local.
        p_imagen = raiz_camvid / particion / Path(imagen).name
        p_mascara = raiz_camvid / f'{particion}annot' / Path(mascara).name
        registros.append({'id': p_imagen.stem, 'imagen': p_imagen,
                          'mascara': p_mascara, 'particion': particion})
verificar_pares(registros, esperado=701)
print('Particiones originales:', dict(Counter(r['particion'] for r in registros)))


## Comprobación de la carga

Se muestran pares originales y los valores de sus máscaras. No se modifica la resolución ni se remapean las etiquetas.


In [ ]:
MAX_ETIQUETA = 11

for registro in registros[:3]:
    imagen, mascara = leer_par(registro)
    valores = np.unique(np.asarray(mascara))
    print(registro['id'], '| tamaño:', imagen.size, '| etiquetas de la máscara:', valores.tolist())
    fig, ejes = plt.subplots(1, 2, figsize=(9, 3))
    ejes[0].imshow(imagen)
    ejes[0].set_title('Imagen original')
    ejes[1].imshow(np.asarray(mascara), cmap='tab20', vmin=0, vmax=MAX_ETIQUETA)
    ejes[1].set_title('Máscara original: índices de etiqueta')
    for eje in ejes:
        eje.axis('off')
    plt.tight_layout()
    plt.show()
